In [1]:
import os

In [2]:
pwd

'c:\\Users\\abhin\\Desktop\\Mlops\\Deep-Learning-Kidney-Tumor-Classification-\\research'

In [3]:
os.chdir("../")

In [4]:
pwd

'c:\\Users\\abhin\\Desktop\\Mlops\\Deep-Learning-Kidney-Tumor-Classification-'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [6]:
from cnnclassifier.constants import config_file_path, params_file_path
from cnnclassifier.utils.common import read_yaml, create_directories

In [ ]:
class configuartionManager:
    def __init__(self, config_file_path = config_file_path, params_file_path = params_file_path):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        create_directories([self.config.artifacts_root])


    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = getattr(self.config, "data_ingestion", None)
        if config is None:
            config = getattr(self.config, "data_ing", None)
        if config is None:
            raise AttributeError("Expected 'data_ingestion' or 'data_ing' section in config")

        create_directories([config.root_dir])
        data_ingestion_config = DataIngestionConfig(
            root_dir=Path(config.root_dir),
            source_URL=config.source_URL,
            local_data_file=Path(config.local_data_file),
            unzip_dir=Path(config.unzip_dir)
        )
        return data_ingestion_config

In [10]:
import os
import zipfile
import gdown
from cnnclassifier import logger
from cnnclassifier.utils.common import get_size

In [ ]:
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_file(self)-> str:
       '''Fetch data from url'''

       try:
           dataset_url = self.config.source_URL
           zip_download_path = self.config.local_data_file
           os.makedirs("artifacts/data_ingestion", exist_ok=True)
           logger.info(f"Downloading data from :[{dataset_url}] into :[{zip_download_path}]")

           if "drive.google.com" in dataset_url and "/file/d/" in dataset_url:
               import re
               match = re.search(r"/file/d/([^/]+)", dataset_url)
               if match:
                   file_id = match.group(1)
                   dataset_url = f"https://drive.google.com/uc?export=download&id={file_id}"

           gdown.download(dataset_url, str(zip_download_path), quiet=False)

           logger.info(f"Downloaded data from :[{dataset_url}] into :[{zip_download_path}] of size :[{get_size(zip_download_path)}]")

       except Exception as e:
             raise e


    def extract_zip_file(self):
        """Extract zip file to specified directory"""
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok=True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            logger.info(f"Extracting zip file :[{self.config.local_data_file}] into dir :[{unzip_path}]")
            zip_ref.extractall(unzip_path)
            logger.info(f"Extracted zip file :[{self.config.local_data_file}] into dir :[{unzip_path}]")

In [26]:
try:
    config = configuartionManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

2026-08-02 19:14:25,436 - cnnclassifier - INFO - yaml file: config\config.yaml loaded successfully
2026-08-02 19:14:25,439 - cnnclassifier - INFO - yaml file: params.yaml loaded successfully
2026-08-02 19:14:25,442 - cnnclassifier - INFO - created directory at: artifacts
2026-08-02 19:14:25,445 - cnnclassifier - INFO - created directory at: artifacts/data_ingestion
2026-08-02 19:14:25,448 - cnnclassifier - INFO - Downloading data from :[https://drive.google.com/uc?export=download&id=1tKknQJ4-0rahmbkq4cVP_b_C3zr4Kc1F] into :[artifacts\data_ingestion\kidney-city-scan-data.zip]


TypeError: download() got an unexpected keyword argument 'fuzzy'